# Phase 3 — Agent skeleton walkthrough

Single supervisory cycle end-to-end with the deterministic `MockLLMClient`. Exercises the full Observer → Optimizer → Critic decomposition plus the pluggable regulatory backend from ADR 009, without touching a real LLM.

This is mechanism validation only — no KPI claims are derived from these runs.

In [ ]:
import json
import numpy as np
from pathlib import Path

from industrial_ai.agents.graph import GraphConfig, run_one_cycle
from industrial_ai.agents.llm_client import MockLLMClient
from industrial_ai.agents.regulatory_backend import build_regulatory_backend
from industrial_ai.twin.column_a import DEFAULT_PARAMETERS

p = DEFAULT_PARAMETERS
NT = p.NT

ss = json.load(open('../data/reference/skogestad_column_a_steady_state.json'))['steady_state']
X_nom = np.array(ss['compositions'] + ss['holdups_kmol'])
print(f'Loaded nominal SS: y_D={X_nom[NT-1]:.4f}, x_B={X_nom[0]:.4f}')

## Scenario A — nominal mock + MPC backend

The mock proposes the nominal product spec `(0.99, 0.01)` on every call. The Critic accepts. The MPC regulatory backend tracks the targets for one supervisor cycle (5 min by default).

In [ ]:
mock = MockLLMClient(policy='nominal')
backend = build_regulatory_backend('mpc')

out = run_one_cycle(
    cycle_index=0, t_min=0.0, X=X_nom,
    LT_kmol_per_min=p.nominal_reflux_L0_kmol_per_min,
    VB_kmol_per_min=p.nominal_boilup_V0_kmol_per_min,
    F_kmol_per_min=p.nominal_feed_F_kmol_per_min,
    zF=0.5, qF=1.0,
    recent_aggregate_iae=0.0,
    llm_client=mock,
    regulatory_backend=backend,
)

print(f'Observer:  y_D={out.state.observer_report.y_D:.4f}  x_B={out.state.observer_report.x_B:.4f}')
print(f'Optimizer: y_D_target={out.state.decision.y_D_target}  x_B_target={out.state.decision.x_B_target}')
print(f'           rationale: {out.state.decision.rationale}')
print(f'Critic:    {out.state.critic_verdict.decision}  ({out.state.critic_verdict.reason})')
print(f'Regulatory backend: {out.regulatory_result.backend_name}  sim_success={out.regulatory_result.simulation.success}')
print(f'After 5-min cycle: y_D={out.regulatory_result.X_final[NT-1]:.4f}  x_B={out.regulatory_result.X_final[0]:.4f}')
print(f'optimizer_rounds={out.optimizer_rounds}  escalated={out.escalated}  wall_clock={out.wall_clock_seconds:.3f}s')

## Scenario B — adaptive mock at the off-nominal F=0.8 / zF=0.45 OP

The off-nominal LV-closed SS has `y_D ≈ 0.72` (composition off-spec by construction). The adaptive mock policy notices the depressed `y_D` in the Observer report and proposes an *interim* target `(0.97, 0.02)` rather than pushing straight to the nominal spec.

This is the Bucket-B target-sequencing path from `docs/kpis.md` §2.3, exercised through the graph without a live LLM.

In [ ]:
from industrial_ai.twin.column_a.operating_window import lookup_lv_ss

X_off = lookup_lv_ss(F=0.8, zF=0.45)
print(f'Off-nominal SS at (F=0.8, zF=0.45): y_D={X_off[NT-1]:.4f}, x_B={X_off[0]:.4f}')

out_off = run_one_cycle(
    cycle_index=0, t_min=0.0, X=X_off,
    LT_kmol_per_min=p.nominal_reflux_L0_kmol_per_min,
    VB_kmol_per_min=p.nominal_boilup_V0_kmol_per_min,
    F_kmol_per_min=0.8, zF=0.45, qF=1.0,
    recent_aggregate_iae=0.0,
    llm_client=MockLLMClient(policy='adaptive'),
    regulatory_backend=backend,
)

print(f'Optimizer: y_D_target={out_off.state.decision.y_D_target}  x_B_target={out_off.state.decision.x_B_target}')
print(f'           rationale: {out_off.state.decision.rationale}')
print(f'Critic:    {out_off.state.critic_verdict.decision}')
print(f'After 5-min cycle: y_D={out_off.regulatory_result.X_final[NT-1]:.4f}  x_B={out_off.regulatory_result.X_final[0]:.4f}')

## Scenario C — same setup with the PID backend (ADR 009 Option B)

Same mock, same plant state, but the regulatory layer is now the C0 PID stack instead of the Linear MPC. The agent graph and the proposal are identical — only the backend changes. This is the "MPC-free deployment" hook from ADR 009.

In [ ]:
pid_backend = build_regulatory_backend('pid')

out_pid = run_one_cycle(
    cycle_index=0, t_min=0.0, X=X_nom,
    LT_kmol_per_min=p.nominal_reflux_L0_kmol_per_min,
    VB_kmol_per_min=p.nominal_boilup_V0_kmol_per_min,
    F_kmol_per_min=1.0, zF=0.5, qF=1.0,
    recent_aggregate_iae=0.0,
    llm_client=MockLLMClient(policy='nominal'),
    regulatory_backend=pid_backend,
)

print(f'Regulatory backend: {out_pid.regulatory_result.backend_name}')
print(f'After 5-min cycle: y_D={out_pid.regulatory_result.X_final[NT-1]:.4f}  x_B={out_pid.regulatory_result.X_final[0]:.4f}')
print(f'sim_success={out_pid.regulatory_result.simulation.success}')

## Scenario D — single supervisory cycle with the live Nemotron endpoint (Schritt 4)

Per ADR 005 amendment 2026-05-28, the agent talks to a native `mlx_lm.server` on the Mac Studio (`gamba@192.168.178.81:8080`) via the `MLXServerLLMClient`. The first-round path uses `reasoning=False` (modal default, fast `/no_think` mode) — `_optimizer_node` flips to `reasoning=True` only if the Critic requests a revision.

This cell requires the server to be running on the Mac Studio. If the endpoint is unreachable the call raises `LLMEndpointUnreachableError` (ADR 010, no silent fallback).


In [ ]:
from industrial_ai.agents.llm_client import MLXServerLLMClient

live_client = MLXServerLLMClient(base_url="http://192.168.178.81:8080/v1")

live_out = run_one_cycle(
    cycle_index=0, t_min=0.0, X=X_nom,
    LT_kmol_per_min=p.nominal_reflux_L0_kmol_per_min,
    VB_kmol_per_min=p.nominal_boilup_V0_kmol_per_min,
    F_kmol_per_min=p.nominal_feed_F_kmol_per_min,
    zF=0.5, qF=1.0,
    recent_aggregate_iae=0.0,
    llm_client=live_client,
    regulatory_backend=backend,  # MPC backend per ADR 009 Option A
)

print(f"Observer:  y_D={live_out.state.observer_report.y_D:.4f}  x_B={live_out.state.observer_report.x_B:.4f}")
print(f"Optimizer (Nemotron): y_D_target={live_out.state.decision.y_D_target}  x_B_target={live_out.state.decision.x_B_target}")
print(f"           rationale: {live_out.state.decision.rationale[:200]}")
print(f"Critic:    {live_out.state.critic_verdict.decision}")
print(f"optimizer_rounds={live_out.optimizer_rounds}  wall_clock={live_out.wall_clock_seconds:.1f} s")
print(f"After 5-min cycle: y_D={live_out.regulatory_result.X_final[NT-1]:.4f}  x_B={live_out.regulatory_result.X_final[0]:.4f}")


## Summary

Four single-cycle runs cover the mechanism surface of the agent skeleton:

- **A:** nominal mock + MPC backend at nominal OP — graph accepts on first round.
- **B:** adaptive mock + MPC backend at off-nominal OP — interim Bucket-B target.
- **C:** nominal mock + PID backend (ADR 009 deployment-economics branch).
- **D:** live Nemotron + MPC backend (ADR 009 Option A primary ladder, ADR 005 amended transport).

Next steps: Phase-3 multi-cycle agent evaluations over the kpis.md §1 nominal and §2.3 / §2.4 off-nominal scenario sets.
